# RS-InternVL: Step 4 Production LoRA Fine-Tuning on Google Colab

This notebook provides a minimal GPU execution environment for fine-tuning the **RS-InternVL3-1B** remote sensing architecture on **BigEarthNet.txt** using Hugging Face PEFT / LoRA.

> **Source of Truth**: The training logic is entirely defined in the repository scripts (`training/train_lora.py` and `training/lora.py`). This notebook executes the repository commands without code duplication.

## 1. Verify CUDA & GPU Hardware

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Device Memory:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Clone Repository or Set Working Directory

In [ ]:
# In Google Colab, clone your GitHub repository (or navigate to workspace directory):
# !git clone https://github.com/<your-org-or-user>/sih2026.git
# %cd sih2026

import os
print(f"Current Working Directory: {os.getcwd()}")

## 3. Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 4. (Optional) Mount Google Drive for Persistent Storage

In [ ]:
# Mount Google Drive to save checkpoints persistently
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/rs_internvl_checkpoints/lora"
    os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
    print(f"Persistent checkpoint directory: {DRIVE_CHECKPOINT_DIR}")
except Exception as e:
    print(f"Running outside Google Colab or Drive mount skipped: {e}")

## 5. Execute Production LoRA Fine-Tuning

Execute `training/train_lora.py` with the production configuration `configs/model/lora.yaml`.
You can customize `--max-train-samples`, `--max-val-samples`, and `--epochs` as needed.

In [ ]:
!python training/train_lora.py \
    --config configs/model/lora.yaml \
    --max-train-samples 1000 \
    --max-val-samples 200 \
    --epochs 3

## 6. Verify Checkpoint Artifacts and Test Reloading

In [ ]:
from pathlib import Path
from training.lora import load_lora_checkpoint

ckpt_dir = Path("checkpoints/lora/best")
print(f"Checkpoint contents in {ckpt_dir}:")
for f in ckpt_dir.iterdir():
    print(f"  - {f.name}")

# Verify full model reconstruction from modular checkpoint
reconstructed_model = load_lora_checkpoint(ckpt_dir, device="cuda" if torch.cuda.is_available() else "cpu")
print("Reconstructed model loaded successfully for inference!")

## 7. (Optional) Sync Checkpoints to Google Drive

In [ ]:
import shutil
if os.path.exists("/content/drive/MyDrive"):
    shutil.copytree("checkpoints/lora", DRIVE_CHECKPOINT_DIR, dirs_exist_ok=True)
    print(f"Checkpoints synced to {DRIVE_CHECKPOINT_DIR}")